# Running Proseg
Runs Proseg transcript-based segmentation on all samples using the `--merscope` preset.

**Inputs:** `transcripts.parquet` per sample  
**Outputs:** `proseg-output.zarr` per sample (in `PROSEG_OUTPUT_DIR`)  
**Config:** fill in `INPUT_DIR` and `PROSEG_OUTPUT_DIR` in the first cell; adjust `CUDA_VISIBLE_DEVICES` as needed

In [ ]:
import os
import pandas as pd
from pathlib import Path

# --- CONFIGURATION: fill in paths before running ---
INPUT_DIR = Path("")        # folder containing per-sample subdirs with transcripts
PROSEG_OUTPUT_DIR = Path("")  # base output dir for proseg results

# Reformat transcripts.parquet to the CSV column schema expected by Proseg (--merscope mode)
for sample in os.listdir(INPUT_DIR):
    print(sample)
    transcripts_df = pd.read_parquet(INPUT_DIR / sample / 'transcripts.parquet')
    new_transcripts_df = pd.DataFrame({
        "feature_name": transcripts_df["gene"].values.astype(str),
        "x_location": transcripts_df["global_x"].values.astype("int64"),
        "y_location": transcripts_df["global_y"].values.astype("int64"),
        "z_location": transcripts_df["global_z"].values.astype("int64"),
        "cell_id": transcripts_df["EntityID"].values.astype("int64"),
        "transcript_id": transcripts_df["transcript_id"].values.astype("int64"),
        "overlaps_nucelus": transcripts_df["overlaps_nucleus"].values.astype(bool),
        "gene": transcripts_df["gene"].values.astype(str)
    })
    new_transcripts_df.to_csv(INPUT_DIR / sample / 'transcripts_for_proseg.csv', index=False)


In [ ]:
import subprocess

# Run Proseg on each sample; adjust CUDA_VISIBLE_DEVICES to your GPU index
for sample in os.listdir(INPUT_DIR):
    command = [
        "proseg",
        "--merscope",
        "--nthreads", "16",
        "--min-qv", "0",
        "--output-path", str(PROSEG_OUTPUT_DIR / sample) + "/",
        "--overwrite",
        "--enforce-connectivity",
        str(INPUT_DIR / sample / 'transcripts_for_proseg.csv')
    ]

    my_env = os.environ.copy()
    my_env["CUDA_VISIBLE_DEVICES"] = "1"
    try:
        print(f"Running Proseg on {sample}...")
        result = subprocess.run(
            command,
            env=my_env,
            check=True,
            capture_output=True,
            text=True
        )
        print("Success!")
    except subprocess.CalledProcessError as e:
        print(f"Error: {e.stderr}")


In [ ]:
#Inspect results
proseg_tx = pd.read_csv(INPUT_DIR / '33156-Slide-22_D1-1' / 'transcripts_for_proseg.csv')


In [15]:
proseg_tx.head()

,feature_name,x_location,y_location,z_location,cell_id,transcript_id,overlaps_nucelus
0,Esrrg,434,1756,39,-1,0,False
1,Esrrg,963,1277,39,367,1,True
2,Esrrg,15,1438,40,-1,2,False
3,Esrrg,563,1292,40,-1,3,False
4,Esrrg,980,1268,40,367,4,True
